In [1]:
!pip install spacy scikit-learn
!python -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 24.0 MB/s  0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [2]:
import spacy
from collections import Counter
from itertools import combinations
from sklearn.svm import SVC
from sklearn.metrics.pairwise import cosine_similarity

nlp = spacy.load("en_core_web_sm")

In [3]:
def get_dependency_path(text, entity1, entity2):

    doc = nlp(text)

    e1 = next((ent for ent in doc.ents
               if ent.text == entity1), None)

    e2 = next((ent for ent in doc.ents
               if ent.text == entity2), None)

    if e1 is None or e2 is None:
        return None

    token1 = e1.root
    token2 = e2.root

    ancestors1 = [token1] + list(token1.ancestors)
    ancestors2 = [token2] + list(token2.ancestors)

    lca = next(
        (t for t in ancestors1 if t in ancestors2),
        None
    )

    if lca is None:
        return None

    path = []

    # Path from first entity to common ancestor
    current = token1

    while current != lca:
        path.append("UP_" + current.dep_)
        current = current.head

    # Path from common ancestor to second entity
    down_path = []
    current = token2

    while current != lca:
        down_path.append("DOWN_" + current.dep_)
        current = current.head

    path.extend(reversed(down_path))

    return path

In [4]:
def subsequence_features(path, max_length=3):

    features = Counter()

    if not path:
        return features

    for length in range(1, min(max_length, len(path)) + 1):

        for indexes in combinations(range(len(path)), length):

            subsequence = tuple(path[i] for i in indexes)

            span = indexes[-1] - indexes[0] + 1

            weight = 0.5 ** (span - length)

            features[subsequence] += weight

    return features

In [5]:
def subsequence_kernel(path1, path2):

    f1 = subsequence_features(path1)
    f2 = subsequence_features(path2)

    common = set(f1) & set(f2)

    return sum(f1[k] * f2[k] for k in common)

In [6]:
training_data = [

    ("Steve Jobs founded Apple.",
     "Steve Jobs", "Apple", "FOUNDED"),

    ("Sundar Pichai leads Google.",
     "Sundar Pichai", "Google", "LEADS"),

    ("Barack Obama was born in Hawaii.",
     "Barack Obama", "Hawaii", "BORN_IN"),

    ("Microsoft acquired LinkedIn.",
     "Microsoft", "LinkedIn", "ACQUIRED"),

    ("Elon Musk founded SpaceX.",
     "Elon Musk", "SpaceX", "FOUNDED"),

    ("Satya Nadella leads Microsoft.",
     "Satya Nadella", "Microsoft", "LEADS")
]

X_train = []
y_train = []

for text, e1, e2, label in training_data:

    path = get_dependency_path(text, e1, e2)

    if path:
        X_train.append(path)
        y_train.append(label)

In [7]:
# Create training kernel matrix

K_train = [
    [subsequence_kernel(p1, p2) for p2 in X_train]
    for p1 in X_train
]

# Train SVM
model = SVC(kernel="precomputed")

model.fit(K_train, y_train)

print("Model trained successfully!")

Model trained successfully!


In [8]:
def predict_relation(text, entity1, entity2):

    path = get_dependency_path(text, entity1, entity2)

    if path is None:
        return "Entities not found"

    K_test = [[
        subsequence_kernel(path, p)
        for p in X_train
    ]]

    prediction = model.predict(K_test)

    print("Entity 1:", entity1)
    print("Entity 2:", entity2)
    print("Dependency Path:", path)
    print("Predicted Relationship:", prediction[0])


# Test example
predict_relation(
    "Elon Musk founded SpaceX.",
    "Elon Musk",
    "SpaceX"
)

'Entities not found'